In [1]:
#@title Upload BoolNet rules file
from google.colab import files
import os, textwrap, csv

# ─────────────────────────────────────────────────────────────
# 1)  Upload / reuse file
# ─────────────────────────────────────────────────────────────
if 'file_rules' in globals() and os.path.exists(file_rules):
    print(f"  Using already loaded file: {file_rules}")
else:
    print(textwrap.dedent("""
        Select your rules file (e.g. 'boolnet_rules_XXX.txt').
        You only need to do this ONCE; the path will be saved in the 'file_rules' variable.
    """))
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError(" You must upload a rules file.")
    file_rules = list(uploaded.keys())[0]
    print(f"  File uploaded and saved in 'file_rules' variable: {file_rules}")

# ─────────────────────────────────────────────────────────────
# 2)  Read rules and calculate N and <K>
# ─────────────────────────────────────────────────────────────
genes   = []        # list of targets
exprs   = []        # list of expressions (factors)

with open(file_rules, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        tgt = row['targets'].strip()
        fac = row['factors'].strip()
        if tgt:                       # discard empty lines
            genes.append(tgt)
            exprs.append(fac)

N = len(genes)

# → translate each expression to extract tokens
tr_table = str.maketrans({'(': ' ', ')': ' ', '&': ' ', '|': ' ', '!': ' '})
k_list = []
for expr in exprs:
    # remove operators and split into tokens
    tokens = expr.translate(tr_table).split()
    # regulators = genes that appear in tokens
    regs   = {tok for tok in tokens if tok in genes}
    k_list.append(len(regs))

K_avg = sum(k_list) / N if N else 0.0

print(f"\n  The network contains {N} nodes.")
print(f"  Average K (mean number of regulators per node): {K_avg:.2f}")


Select your rules file (e.g. 'boolnet_rules_XXX.txt').
You only need to do this ONCE; the path will be saved in the 'file_rules' variable.



Saving nsclc_9_nodes.txt to nsclc_9_nodes.txt
  File uploaded and saved in 'file_rules' variable: nsclc_9_nodes.txt

  The network contains 9 nodes.
  Average K (mean number of regulators per node): 1.89


In [2]:
# @title Enumerate and simulate all equivalence classes (update-digraphs)

import pandas as pd, numpy as np, re, ast, csv, os, itertools, textwrap, os, shutil
from collections import defaultdict, Counter

# ────────────────────────────────────────────────────────────────────────────────
# Colab detection (for upload/download)
# ────────────────────────────────────────────────────────────────────────────────
try:
    from google.colab import files  # type: ignore
    _IN_COLAB = True
except Exception:
    _IN_COLAB = False
    class _FilesShim:
        def upload(self):
            print(" Run this in Colab to use 'files.upload()' or set 'file_rules' manually.");
            return {}
        def download(self, path):
            print(f" Manual download: {os.path.abspath(path)}")
    files = _FilesShim()

# ────────────────────────────────────────────────────────────────────────────────
# Utilities
# ────────────────────────────────────────────────────────────────────────────────
_TOKEN_RE = re.compile(r"\b[\w]+\b")
_KW = {"and","or","not","true","false","True","False"}

def boolnet_to_py(expr: str) -> str:
    """Convert BoolNet operators to Python."""
    expr = re.sub(r'!', ' not ', expr)
    expr = re.sub(r'&', ' and ', expr)
    expr = re.sub(r'\|', ' or ', expr)
    expr = re.sub(r'\bTRUE\b',  'True',  expr, flags=re.I)
    expr = re.sub(r'\bFALSE\b', 'False', expr, flags=re.I)
    return re.sub(r'\s+', ' ', expr).strip()

def load_rules(fname: str) -> dict:
    """Load BoolNet rules from CSV (Targets,Expr) or 'Node: expr' file."""
    rules = {}
    with open(fname, encoding="utf-8") as fh:
        first = fh.readline().strip(); fh.seek(0)
        if ":" in first and not first.lower().startswith(("targets", "target")):
            for line in fh:
                if ":" not in line: continue
                node, expr = map(str.strip, line.split(":", 1))
                rules[node] = boolnet_to_py(expr)
        else:
            for row in csv.reader(fh, delimiter=','):
                if not row: continue
                node, expr, *rest = row
                if node.lower().startswith(("targets","target")): continue
                rules[node.strip()] = boolnet_to_py(str(expr).strip())
    return rules

def extract_deps_from_rules(rules: dict) -> dict:
    """Extract dependencies by tokenizing pythonized expressions."""
    nodes = set(rules.keys())
    deps = {}
    for tgt, expr in rules.items():
        toks = _TOKEN_RE.findall(expr)
        deps[tgt] = [t for t in toks if t in nodes and t != tgt]
    return deps

def infer_network_name(path: str) -> str:
    """Infer a safe network name from the rules filename."""
    base = os.path.splitext(os.path.basename(path))[0]
    safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', base).strip('_')
    return safe or "network"

# ────────────────────────────────────────────────────────────────────────────────
# 0) Locate rules file
# ────────────────────────────────────────────────────────────────────────────────
if 'file_rules' not in globals() or not os.path.exists(str(globals().get('file_rules', ''))):
    print("  'file_rules' not found. Please upload your rules file…")
    up = files.upload()
    if not up:
        raise RuntimeError(" You must upload a rules file.")
    file_rules = list(up.keys())[0]

print(f"  Using rules: {file_rules}")
rules = load_rules(file_rules)
nodes = list(rules.keys())
print(" Detected nodes:", nodes)

# ────────────────────────────────────────────────────────────────────────────────
# Output folder and filenames
# ────────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = "determining_distinct_update_schemes"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NETWORK_NAME = infer_network_name(file_rules)
ZIP_BASENAME = f"{NETWORK_NAME}_determining_distinct_update_schemes"   # without .zip (added by make_archive)

equiv_csv = os.path.join(OUTPUT_DIR, "equiv_classes.csv")
attr_csv  = os.path.join(OUTPUT_DIR, "Attractor_Results_equiv_class.csv")
report_txt= os.path.join(OUTPUT_DIR, "summary_attractors_equiv_class.txt")

# ────────────────────────────────────────────────────────────────────────────────
# 1) Enumerate equivalence classes (update-digraphs)
# ────────────────────────────────────────────────────────────────────────────────
deps = extract_deps_from_rules(rules)
edges = [(u, t) for t, Ss in deps.items() for u in Ss]

def generate_block_schedules(nodes):
    """Generate all surjective block-sequential schedules over 1..m."""
    n = len(nodes)
    for m in range(1, n+1):
        rng = range(1, m+1)
        for vals in itertools.product(rng, repeat=n):
            if set(vals) == set(rng):
                yield dict(zip(nodes, vals))

def signature(schedule):
    """P/N signature per edge: P if u>=v, N if u<v."""
    return tuple(sorted(
        (u, v, "P" if schedule[u] >= schedule[v] else "N")
        for u, v in edges
    ))

print("  Enumerating equivalence classes…")
groups = {}
total_sched = 0
for sched in generate_block_schedules(nodes):
    total_sched += 1
    sig = signature(sched)
    if sig not in groups:
        groups[sig] = sched

print(f" Total brute-force schedules: {total_sched}")
print(f"⧉ Equivalence classes (update-digraphs): {len(groups)}")

with open(equiv_csv, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["Scheme", *nodes])
    for i, sched in enumerate(groups.values(), 1):
        w.writerow([f"S_{i}", *[sched[n] for n in nodes]])
print(f" CSV saved: {os.path.abspath(equiv_csv)}")

# ────────────────────────────────────────────────────────────────────────────────
# 2) Simulate boolean network for each schedule and extract attractors
# ────────────────────────────────────────────────────────────────────────────────
import pandas as pd, numpy as np

def simulate_network(rules: dict, upd_vec, nodes):
    """Transition map over all states (2^n) applying the block schedule."""
    table = []
    n = len(nodes)
    steps = sorted(set(upd_vec))
    for i in range(2**n):
        bin_state = format(i, f"0{n}b")
        current = {node: int(bin_state[idx]) for idx, node in enumerate(nodes)}
        for step in steps:
            nxt = current.copy()
            to_upd = [node for idx, node in enumerate(nodes) if upd_vec[idx] == step]
            for node in to_upd:
                env = {k: bool(v) for k, v in current.items()}
                nxt[node] = int(eval(rules[node], {}, env))
            current = nxt
        table.append(f"{bin_state} => " + ''.join(str(current[n]) for n in nodes))
    return table

def _normalize_cycle(cyc):
    idx = min(range(len(cyc)), key=lambda i: cyc[i])
    return cyc[idx:] + cyc[:idx]

def find_attractors(table):
    mp = {row.split(' => ')[0]: row.split(' => ')[1] for row in table}
    attr, basin = {}, {}
    for start in mp:
        vis, traj, cur = [], [], start
        while cur not in vis:
            vis.append(cur); traj.append(cur); cur = mp[cur]
        cyc = tuple(vis[vis.index(cur):])
        cyc_norm = _normalize_cycle(cyc)
        attr.setdefault(cyc_norm, set()).update(vis)
        for s in traj: basin[s] = cyc_norm
    return attr, basin

df_sched = pd.read_csv(equiv_csv)

if "Schedule" in df_sched.columns:
    def extract_vec(row): return ast.literal_eval(row["Schedule"])
elif set(nodes).issubset(df_sched.columns):
    def extract_vec(row): return [int(row[node]) for node in nodes]
else:
    raise RuntimeError(
        "The schedules CSV must have a 'Schedule' column with lists, "
        "or separate columns for each node."
    )

print(f"  Simulating {len(df_sched)} schedule(s)…")
results = []
for i, row in df_sched.iterrows():
    upd_vec = extract_vec(row)
    if len(upd_vec) != len(nodes):
        print(f"  Row {i} skipped (length {len(upd_vec)} ≠ {len(nodes)})")
        continue
    trans = simulate_network(rules, upd_vec, nodes)
    attract, bas = find_attractors(trans)
    for cyc, bs in sorted(attract.items(), key=lambda x: len(x[1]), reverse=True):
        results.append({
            "Scheme": row.get("Scheme", f"S_{i+1}"),
            "Update Vector": upd_vec,
            "Attractor": cyc,
            "Basin Size": len(bs)
        })

pd.DataFrame(results).to_csv(attr_csv, index=False, sep=';')
print(f" Results saved: {os.path.abspath(attr_csv)}")

# ────────────────────────────────────────────────────────────────────────────────
# 3) Summary report
# ────────────────────────────────────────────────────────────────────────────────
df = pd.read_csv(attr_csv, sep=';')
df.columns = [c.strip() for c in df.columns]

def find_col(pat, cols):
    rx = re.compile(pat, flags=re.I)
    for c in cols:
        if rx.search(c): return c
    raise KeyError(f"Could not find a column containing '{pat}'")

col_scheme     = find_col(r"scheme", df.columns)
col_attractor  = find_col(r"attractor", df.columns)
col_basin_size = find_col(r"basin\s*size", df.columns)

df["_AttrParsed"] = df[col_attractor].apply(ast.literal_eval)

schemes = {}
fixed_stats = defaultdict(lambda: {"count":0, "basins":[]})
cycle_stats = defaultdict(lambda: {"count":0, "basins":[]})

for _, row in df.iterrows():
    sch = row[col_scheme]
    attr = row["_AttrParsed"]
    bs   = int(row[col_basin_size])
    rec = schemes.setdefault(sch, {"fixed": [], "cycles": []})
    if len(attr) == 1:
        rec["fixed"].append((attr, bs))
        fixed_stats[attr]["count"] += 1
        fixed_stats[attr]["basins"].append(bs)
    else:
        rec["cycles"].append((attr, bs))
        cycle_stats[attr]["count"] += 1
        cycle_stats[attr]["basins"].append(bs)

total_schemes = len(schemes)
only_fixed    = sum(1 for v in schemes.values() if not v["cycles"])
with_cycles   = sum(1 for v in schemes.values() if v["cycles"])
pct_fixed     = (only_fixed/total_schemes*100) if total_schemes else 0.0
pct_cycles    = (with_cycles/total_schemes*100) if total_schemes else 0.0
total_cycles  = sum(info["count"] for info in cycle_stats.values())

lines = []
lines.append("="*70)
lines.append(f"1- Total number of distinct update schemes found: {total_schemes}\n")
lines.append(f"2- Number of schemes that only presented fixed points: {only_fixed}  ({pct_fixed:6.2f} %)")
lines.append(f"3- Number of schemes that presented cycles: {with_cycles}  ({pct_cycles:6.2f} %)\n")

lines.append("4- Equal fixed points found:\n")
for idx, (attr, info) in enumerate(
        sorted(fixed_stats.items(), key=lambda x: x[1]["count"], reverse=True), start=1):
    avg_basin = sum(info["basins"])/len(info["basins"]) if info["basins"] else 0.0
    state = list(attr) if isinstance(attr, (list, tuple)) else [attr]
    lines.append(f"    {idx:2d}) Found {info['count']:>4} times: {state}, Average basin size: {avg_basin:.2f}")

lines.append("\n5- Equal limit cycles found (Top 10):\n")
for idx, (attr, info) in enumerate(
        sorted(cycle_stats.items(), key=lambda x: x[1]["count"], reverse=True)[:10], start=1):
    cnt = info["count"]
    pct = cnt/total_cycles*100 if total_cycles else 0
    avg_basin = sum(info["basins"])/len(info["basins"]) if info["basins"] else 0.0
    lines.append(f"    {idx:2d}) Found {cnt:>4} times ({pct:6.2f} %) : {attr} | Average basin size: {avg_basin:.2f}")

lines.append("\n6- Schemes by number of limit cycles:")
cycles_per_scheme = Counter(len(v["cycles"]) for v in schemes.values() if v["cycles"])
for k in sorted(cycles_per_scheme):
    lines.append(f"   • Schemes with {k} cycle(s): {cycles_per_scheme[k]}")

lines.append("\n7- Total limit cycles found by length:")
cycles_total_by_length = Counter()
for attr, info in cycle_stats.items():
    L = len(attr) if isinstance(attr, (list, tuple)) else 1
    cycles_total_by_length[L] += info["count"]
for L in sorted(cycles_total_by_length):
    lines.append(f"   • Cycles of length {L}: {cycles_total_by_length[L]}")

lines.append("="*70)

report = "\n".join(lines)
with open(report_txt, "w") as f:
    f.write(report)
print(report)
print(f" Report saved: {os.path.abspath(report_txt)}")

# ────────────────────────────────────────────────────────────────────────────────
# 4) Create ZIP: <NETWORK_NAME>_determining_distinct_update_schemes.zip
#    (includes the whole determining_distinct_update_schemes/ folder)
# ────────────────────────────────────────────────────────────────────────────────
# To include the directory itself in the ZIP, use base_dir=OUTPUT_DIR with root_dir='.'
zip_path = shutil.make_archive(ZIP_BASENAME, 'zip', root_dir='.', base_dir=OUTPUT_DIR)
print(f"🗜️ ZIP created: {os.path.abspath(zip_path)}")

if _IN_COLAB:
    files.download(zip_path)

print("\n Done. Files generated inside:", os.path.abspath(OUTPUT_DIR))
print("   -", os.path.abspath(equiv_csv))
print("   -", os.path.abspath(attr_csv))
print("   -", os.path.abspath(report_txt))
print("   ZIP:", os.path.abspath(zip_path))


  Using rules: nsclc_9_nodes.txt
 Detected nodes: ['miR_145', 'Sp1', 'MALAT1', 'BMI1', 'KLF4', 'p53', 'p53_A', 'p53_K', 'E2F1']
  Enumerating equivalence classes…
 Total brute-force schedules: 7087261
⧉ Equivalence classes (update-digraphs): 10632
 CSV saved: /content/determining_distinct_update_schemes/equiv_classes.csv
  Simulating 10632 schedule(s)…
 Results saved: /content/determining_distinct_update_schemes/Attractor_Results_equiv_class.csv
1- Total number of distinct update schemes found: 10632

2- Number of schemes that only presented fixed points: 7356  ( 69.19 %)
3- Number of schemes that presented cycles: 3276  ( 30.81 %)

4- Equal fixed points found:

     1) Found 10632 times: ['011110001'], Average basin size: 441.19
     2) Found 10632 times: ['100001010'], Average basin size: 23.25
     3) Found 10632 times: ['100001100'], Average basin size: 23.25

5- Equal limit cycles found (Top 10):

     1) Found 2658 times ( 67.29 %) : ('100001000', '100001110') | Average basin siz

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 Done. Files generated inside: /content/determining_distinct_update_schemes
   - /content/determining_distinct_update_schemes/equiv_classes.csv
   - /content/determining_distinct_update_schemes/Attractor_Results_equiv_class.csv
   - /content/determining_distinct_update_schemes/summary_attractors_equiv_class.txt
   ZIP: /content/nsclc_9_nodes_determining_distinct_update_schemes.zip
